# Word2Vec + MLP Classifier

This notebook uses the continuous skip-gram architecture from the [official TensorFlow Word2Vec Tutorial](https://www.tensorflow.org/text/tutorials/word2vec) to train word embeddings from scratch. 

After extracting the learned embeddings, we average the word vectors for each sequence to create sentence representations and classify them using a Multilayer Perceptron (MLP) built with Keras.

In [1]:
import io
import re
import string
import tqdm
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from sklearn.model_selection import train_test_split

# Setup standard seeds and configurations
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE
tf.random.set_seed(SEED)

## 1. Load the Dataset
Loading `../data/train_noDuplicates.csv`. Ensure the column names `'text'` and `'label'` match your dataset.

In [2]:
# Load the explicitly named dataset
df = pd.read_csv('/kaggle/input/datasets/juliaduboisas/train-noduplicates/../data/train_noDuplicates.csv')

# Drop any rows with missing data in our target columns
df = df.dropna(subset=['resp_text', 'clarity'])

text_data = df['resp_text'].astype(str).tolist()
labels = df['clarity'].tolist()

print(f"Loaded {len(text_data)} samples.")

Loaded 18285 samples.


## 2. Text Preprocessing & Vectorization
We standardize, tokenize, and map words to integer IDs using Keras `TextVectorization`.

In [3]:
# Define vectorization parameters
vocab_size = 10000
sequence_length = 50

# Custom standardizer to lowercase and remove punctuation
def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    return tf.strings.regex_replace(lowercase, '[%s]' % re.escape(string.punctuation), '')

vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length
)

# Build the vocabulary based on the dataset
vectorize_layer.adapt(text_data)
inverse_vocab = vectorize_layer.get_vocabulary()
print(f"Vocabulary size: {len(inverse_vocab)}")

# Vectorize all text data into sequences of integers
text_vectorized = vectorize_layer(text_data).numpy()

I0000 00:00:1790793577.288032      57 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790793577.290929      57 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Vocabulary size: 10000


## 3. Generate Skip-grams & Negative Samples
Generating `(target, context)` pairs to train the Word2Vec model.

In [4]:
def generate_training_data(sequences, window_size, num_ns, vocab_size, seed):
    targets, contexts, w2v_labels = [], [], []
    
    for sequence in tqdm.tqdm(sequences):
        positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
              sequence,
              vocabulary_size=vocab_size,
              window_size=window_size,
              negative_samples=0,
              seed = SEED)
        
        for target_word, context_word in positive_skip_grams:
            context_class = tf.expand_dims(tf.constant([context_word], dtype="int64"), 1)
            
            negative_sampling_candidates, _, _ = tf.random.log_uniform_candidate_sampler(
                true_classes=context_class,
                num_true=1,
                num_sampled=num_ns,
                unique=True,
                range_max=vocab_size,
                seed=seed,
                name="negative_sampling")
            
            context = tf.concat([tf.squeeze(context_class, 1), negative_sampling_candidates], 0)
            label = tf.constant([1] + [0]*num_ns, dtype="int64")
            
            targets.append(target_word)
            contexts.append(context)
            w2v_labels.append(label)

    return targets, contexts, w2v_labels

# Generate Word2Vec training data
window_size = 2
num_ns = 4

targets, contexts, w2v_labels = generate_training_data(
    sequences=text_vectorized, 
    window_size=window_size, 
    num_ns=num_ns, 
    vocab_size=vocab_size, 
    seed=SEED
)

targets = np.array(targets)
contexts = np.array(contexts)
w2v_labels = np.array(w2v_labels)

# Build performant tf.data.Dataset
BATCH_SIZE = 1024
BUFFER_SIZE = 10000
dataset = tf.data.Dataset.from_tensor_slices(((targets, contexts), w2v_labels))
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)
dataset = dataset.cache().prefetch(buffer_size=AUTOTUNE)

100%|██████████| 18285/18285 [55:32<00:00,  5.49it/s]  


## 4. Build and Train the Subclassed Word2Vec Model
This defines the `Word2Vec` model and computes the dot products between target and context embeddings.

In [5]:
class Word2Vec(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, num_ns):
        super(Word2Vec, self).__init__()
        self.target_embedding = layers.Embedding(vocab_size,
                                                 embedding_dim,
                                                 input_length=1,
                                                 name="w2v_embedding")
        self.context_embedding = layers.Embedding(vocab_size,
                                                  embedding_dim,
                                                  input_length=num_ns+1)

    def call(self, pair):
        target, context = pair
        if len(target.shape) == 2:
            target = tf.squeeze(target, axis=1)
            
        word_emb = self.target_embedding(target)
        context_emb = self.context_embedding(context)
        
        dots = tf.einsum('be,bce->bc', word_emb, context_emb)
        return dots

embedding_dim = 128
word2vec = Word2Vec(vocab_size, embedding_dim, num_ns)

word2vec.compile(optimizer='adam',
                 loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])

# Train the Word2Vec model 
word2vec.fit(dataset, epochs=5)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


  33/3052 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - accuracy: 0.2502 - loss: 1.6071 

I0000 00:00:1790797154.734819     132 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3052/3052 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.5763 - loss: 1.0881
Epoch 2/5
3052/3052 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.6555 - loss: 0.8988
Epoch 3/5
3052/3052 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.6777 - loss: 0.8395
Epoch 4/5
3052/3052 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.6915 - loss: 0.8024
Epoch 5/5
3052/3052 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.7018 - loss: 0.7751


## 5. Feature Engineering: Word Vector Averaging
We extract the learned weights from the `'w2v_embedding'` layer and average the word vectors for the tokens in each sentence to create a single vector representation for the MLP.

In [8]:
from sklearn.preprocessing import LabelEncoder

# Retrieve the learned word embeddings
weights = word2vec.get_layer('w2v_embedding').get_weights()[0]

def get_sentence_vector(sequence):
    valid_words = [w for w in sequence if w != 0]
    if len(valid_words) == 0:
        return np.zeros(embedding_dim)
    
    embeddings = [weights[word] for word in valid_words]
    return np.mean(embeddings, axis=0)

# Create final feature matrix X
X = np.array([get_sentence_vector(seq) for seq in text_vectorized])

# Encode the string labels into numerical integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

print(f"Feature matrix X shape: {X.shape}")
print(f"Target array y shape: {y.shape}")
print(f"Classes mapped: {dict(enumerate(label_encoder.classes_))}")

Feature matrix X shape: (18285, 128)
Target array y shape: (18285,)
Classes mapped: {0: np.str_('c1'), 1: np.str_('c234'), 2: np.str_('c5')}


## 6. Classification with a Multilayer Perceptron (MLP)
We dynamically configure the MLP for binary or multi-class classification based on the number of unique labels, then compile and train the network.

In [9]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

# Determine if binary or multi-class classification
num_classes = len(np.unique(y))
is_binary = (num_classes == 2)

# Build the MLP architecture
mlp_model = Sequential([
    layers.Dense(64, activation='relu', input_shape=(embedding_dim,)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3)
])

# Add the correct output layer
if is_binary:
    mlp_model.add(layers.Dense(1, activation='sigmoid'))
    loss_fn = 'binary_crossentropy'
else:
    mlp_model.add(layers.Dense(num_classes, activation='softmax'))
    loss_fn = 'sparse_categorical_crossentropy'

# Compile the model
mlp_model.compile(optimizer='adam',
                  loss=loss_fn,
                  metrics=['accuracy'])

mlp_model.summary()

# Train the MLP
print("\nTraining MLP Classifier...")
history = mlp_model.fit(
    X_train, y_train, 
    epochs=15, 
    batch_size=32, 
    validation_split=0.2,
    verbose=1
)

# Evaluate on the test set
print("\nEvaluating on Test Set...")
test_loss, test_accuracy = mlp_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,435 (40.76 KB)

 Trainable params: 10,435 (40.76 KB)

 Non-trainable params: 0 (0.00 B)


Training MLP Classifier...
Epoch 1/15
366/366 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.3730 - loss: 1.0887 - val_accuracy: 0.4019 - val_loss: 1.0783
Epoch 2/15
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3926 - loss: 1.0799 - val_accuracy: 0.4050 - val_loss: 1.0752
Epoch 3/15
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4021 - loss: 1.0739 - val_accuracy: 0.4129 - val_loss: 1.0718
Epoch 4/15
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4085 - loss: 1.0702 - val_accuracy: 0.4193 - val_loss: 1.0698
Epoch 5/15
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4110 - loss: 1.0672 - val_accuracy: 0.4197 - val_loss: 1.0674
Epoch 6/15
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4169 - loss: 1.0647 - val_accuracy: 0.4180 - val_loss: 1.0670
Epoch 7/15
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4190 - loss: 1.0613 - val_accuracy: 0.4344 - val_loss: 1.0670
Epoch 8/15
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4269 - lo